# Approach: Min-K% and XGBoost
In this approach, I use the same pipeline as in Milestone 1 to load the data, extract features, and process in batches. However, some additions are made, including a new min-k% feature (members will tend to have a higher model confidence on the most surprising, least likely tokens) and an XGBoost classifier to see if better performance is possible. Cross-model difference and ratio features were added, since, after some examination feature importance, it was clear that the 360m model's feature were adding more to the classifier.    
Note: I also tried to use ReCaLL, however, this feature did not end up contributing to the overall performance of the classifier models, but rather, added unnecessary noise. I've taken this feature out retroactively, but this was among the features extracted initially.

In [2]:
# Core imports
import wandb
from huggingface_hub import HfApi, notebook_login
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

# Set device to use gpu if possible.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cpu


In [28]:
# Tracking results
# HF
notebook_login()

# W&B login
wandb.login(key="your_key")

# Initialise a W&B run
wandb.init(
    project="mia-tcob",
    name="smollm2-135m-360m-recall-mink",
)

# Log all hyperparameters and settings for reproducibility
wandb.config.update({
    # Feature selection
    "mi_k": 15,
    # XGBoost
    "xgb_n_estimators": 200,
    "xgb_learning_rate": 0.05,
    "xgb_max_depth": 4,
    "xgb_subsample": 0.8,
    "xgb_colsample_bytree": 0.8,
    # Pipeline
    "batch_size": 500,
    "max_token_length": 512,
    "min_k_values": [10, 20, 30],
    # Models
    "target_models":["UBC-SLIME/colx_531_smollm2-135m",
                     "UBC-SLIME/colx_531_smollm2-360m"],
    "dataset": "UBC-SLIME/colx_531_group_project",
})

print("W&B run initialised:", wandb.run.name)
print("W&B project:", wandb.run.project)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Logistic Regression/accuracy,▁▆▁█
Logistic Regression/auc,▁▇▁█
Logistic Regression/tpr_at_fpr_01,▂▇▁█
Random Forest/accuracy,▁▇▃█
Random Forest/auc,▁█▃█
Random Forest/tpr_at_fpr_01,▁█▁▇
XGBoost/accuracy,▁▅▅█
XGBoost/auc,▁▅▅█
XGBoost/tpr_at_fpr_01,▁▅▅█
Logistic Regression/accuracy,0.8208
Logistic Regression/auc,0.89023


W&B run initialised: smollm2-135m-360m-recall-mink
W&B project: mia-tcob


In [4]:
# Load dataset and models
print("Loading dataset...")
dataset = load_dataset("UBC-SLIME/colx_531_group_project")

print("Loading models...")
model_135m = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")
model_360m = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m")
tokenizer = AutoTokenizer.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")

# Move models to device
model_135m.to(device)
model_360m.to(device)
model_135m.eval()
model_360m.eval()

print(f"Dataset: {len(dataset['train'])} train, {len(dataset['validation'])} validation samples")
print("Models loaded: 135M and 360M parameters")

Loading dataset...


train.json:   0%|          | 0.00/102M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/30.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset: 50000 train, 10000 validation samples
Models loaded: 135M and 360M parameters


# Feature Extraction
Function from Milestone 1 expanded upon to calculate min-k% and ReCaLL.

In [7]:
def compute_perplexity_and_features(model, tokenizer, text, max_length=512):
    """
    Compute perplexity and other features that might indicate memorization.

    Returns:
        dict with perplexity, loss, confidence metrics, Min-K% features

    """
    inputs = tokenizer(text, return_tensors="pt", max_length=max_length,
                       truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        logits = outputs.logits

        # Lower loss, model has likely seen this pattern before
        loss = outputs.loss.item()

        # Low perplexity, more likely seen before
        perplexity = torch.exp(outputs.loss).item()

        shift_logits = logits[..., :-1, :]
        shift_labels = inputs["input_ids"][..., 1:]
        token_probs = F.softmax(shift_logits, dim=-1)

        # Probability of the actual token at each position
        actual_token_probs = token_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

        # Average confidence across all tokens
        mean_token_prob = actual_token_probs.mean().item()

        # Lowest / highest single-token confidence
        min_token_prob = actual_token_probs.min().item()
        max_token_prob = actual_token_probs.max().item()

        # Lower std, uniformly high confidence, more likely seen before
        std_token_prob = actual_token_probs.std().item()

        # Higher entropy, more uncertain, less likely seen before
        token_entropies = -(token_probs * torch.log(token_probs + 1e-10)).sum(dim=-1)
        mean_entropy = token_entropies.mean().item()

        # High top-k mass, confident and likely seen before
        top_k_probs, _ = torch.topk(token_probs, k=10, dim=-1)
        top_k_mass = top_k_probs.sum(dim=-1).mean().item()

        # Low ratio, few uncertain tokens, likely seen before
        low_conf_tokens = (actual_token_probs < 0.1).sum().item()
        low_conf_ratio = low_conf_tokens / actual_token_probs.shape[1]

        # Min-K% features
        # Average probability of the bottom K% of tokens.
        # Members have higher confidence even on their hardest tokens.
        sorted_probs = actual_token_probs.sort(dim=-1).values  # ascending
        n_tokens = sorted_probs.shape[1]

        k10 = max(1, int(n_tokens * 0.10))
        min_k10_prob = sorted_probs[:, :k10].mean().item()

        k20 = max(1, int(n_tokens * 0.20))
        min_k20_prob = sorted_probs[:, :k20].mean().item()

        k30 = max(1, int(n_tokens * 0.30))
        min_k30_prob = sorted_probs[:, :k30].mean().item()

    result = {
        "perplexity":      perplexity,
        "loss":            loss,
        "mean_token_prob": mean_token_prob,
        "min_token_prob":  min_token_prob,
        "max_token_prob":  max_token_prob,
        "std_token_prob":  std_token_prob,
        "mean_entropy":    mean_entropy,
        "top_k_mass":      top_k_mass,
        "low_conf_ratio":  low_conf_ratio,
        "min_k10_prob":    min_k10_prob,
        "min_k20_prob":    min_k20_prob,
        "min_k30_prob":    min_k30_prob,
        "num_tokens":      n_tokens,
    }

    return result

In [9]:
def analyze_sample_batch(model, tokenizer, texts, labels, model_name="model"):
    """
    Analyze a batch of texts and return features for membership inference.

    """
    results = []

    for text, label in tqdm(zip(texts, labels), total=len(texts),
                            desc=f"Analyzing with {model_name}"):
        try:
            features = compute_perplexity_and_features(
                model, tokenizer, text, recall_prefix=recall_prefix
            )
            features["label"] = label
            features["text_length"] = len(text)
            results.append(features)
        except Exception as e:
            print(f"Error processing text (len={len(text)}): {str(e)[:100]}")
            results.append({
                "perplexity": float("inf"),
                "loss": float("inf"),
                "mean_token_prob": 0,
                "min_k10_prob": 0,
                "min_k20_prob": 0,
                "min_k30_prob": 0,
                "recall_score": float("inf"),
                "label": label,
                "text_length": len(text),
            })

    return pd.DataFrame(results)

In [10]:
def process_dataset_in_batches(model, tokenizer, texts, labels,
                               model_name="model", batch_size=1000):
    """
    Process large dataset in batches for memory efficiency.
    """
    all_results = []
    print(f"Processing {len(texts)} samples in batches of {batch_size}...")

    for i in range(0, len(texts), batch_size):
        end_idx     = min(i + batch_size, len(texts))
        batch_texts  = texts[i:end_idx]
        batch_labels = labels[i:end_idx]

        print(f"Batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}: "
              f"{end_idx - i} samples")

        batch_results = analyze_sample_batch(
            model, tokenizer, batch_texts, batch_labels,
            model_name, recall_prefix=recall_prefix
        )
        all_results.append(batch_results)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.concat(all_results, ignore_index=True)
    print(f"Completed processing {len(final_results)} samples with {model_name}")
    return final_results

# Dataset Preparation
Load and prepare the full training and validation datasets

In [33]:
# Load full dataset splits
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f"Training set: {len(train_df)} samples")
print(f"Validation set: {len(val_df)} samples")
print(f"Test set: {len(test_df)} samples")
print(f"Training - Members: {sum(train_df['is_member'])}, Non-members: {len(train_df) - sum(train_df['is_member'])}")
print(f"Validation - Members: {sum(val_df['is_member'])}, Non-members: {len(val_df) - sum(val_df['is_member'])}")

# Extract texts and labels
train_texts = train_df['text'].tolist()
train_labels = train_df['is_member'].tolist()
val_texts = val_df['text'].tolist()
val_labels = val_df['is_member'].tolist()
test_texts = test_df['text'].tolist()
test_labels = [0] * len(test_texts)  # Dummy labels for test set

Training set: 50000 samples
Validation set: 10000 samples
Test set: 15000 samples
Training - Members: 25000, Non-members: 25000
Validation - Members: 5000, Non-members: 5000


In [13]:
# Process training data
print("Processing training data with 135M model...")
train_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, train_texts, train_labels, "135M", batch_size=500
)

print("\nProcessing training data with 360M model...")
train_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, train_texts, train_labels, "360M", batch_size=500
)

print(f"\nTraining data processing complete!")
print(f"135M results: {train_results_135m.shape}")
print(f"360M results: {train_results_360m.shape}")

Processing training data with 135M model...
Processing 50000 samples in batches of 500...
Batch 1/100: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:23<00:00, 21.73it/s]


Batch 2/100: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:21<00:00, 23.15it/s]


Batch 3/100: 500 samples


...

Batch 99/100: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:23<00:00, 21.42it/s]


Batch 100/100: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:23<00:00, 21.57it/s]

Completed processing 50000 samples with 360M

Training data processing complete!
135M results: (50000, 16)
360M results: (50000, 16)


In [14]:
# Process validation data
print("Processing validation data with 135M model...")
val_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, val_texts, val_labels, "135M", batch_size=500
)

print("\nProcessing validation data with 360M model...")
val_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, val_texts, val_labels, "360M", batch_size=500
)

print(f"\nValidation data processing complete!")
print(f"135M results: {val_results_135m.shape}")
print(f"360M results: {val_results_360m.shape}")

Processing validation data with 135M model...
Processing 10000 samples in batches of 500...
Batch 1/20: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:22<00:00, 21.78it/s]


Batch 2/20: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:22<00:00, 21.90it/s]


Batch 3/20: 500 samples


...

Batch 19/20: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:23<00:00, 20.92it/s]


Batch 20/20: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:23<00:00, 21.20it/s]

Completed processing 10000 samples with 360M

Validation data processing complete!
135M results: (10000, 16)
360M results: (10000, 16)


In [15]:
# Process test data
print("Processing test data with 135M model...")
test_results_135m = process_dataset_in_batches(
    model_135m, tokenizer, test_texts, test_labels, "135M", batch_size=500
)

print("\nProcessing test data with 360M model...")
test_results_360m = process_dataset_in_batches(
    model_360m, tokenizer, test_texts, test_labels, "360M", batch_size=500
)

print(f"\nTest data processing complete!")
print(f"135M results: {test_results_135m.shape}")
print(f"360M results: {test_results_360m.shape}")

Processing test data with 135M model...
Processing 15000 samples in batches of 500...
Batch 1/30: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:22<00:00, 21.86it/s]


Batch 2/30: 500 samples


Analyzing with 135M: 100%|██████████| 500/500 [00:22<00:00, 22.07it/s]


Batch 3/30: 500 samples


...

Batch 29/30: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:23<00:00, 21.05it/s]


Batch 30/30: 500 samples


Analyzing with 360M: 100%|██████████| 500/500 [00:23<00:00, 21.26it/s]

Completed processing 15000 samples with 360M

Test data processing complete!
135M results: (15000, 16)
360M results: (15000, 16)


In [16]:
# Save all extracted features to disk immediately after extraction
# This makes feature extraction and classifier training fully independent
# sessions.

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/mia_features/"

import os
os.makedirs(SAVE_DIR, exist_ok=True)

train_results_135m.to_csv(f"{SAVE_DIR}train_features_135m.csv", index=False)
train_results_360m.to_csv(f"{SAVE_DIR}train_features_360m.csv", index=False)
val_results_135m.to_csv(  f"{SAVE_DIR}val_features_135m.csv",   index=False)
val_results_360m.to_csv(  f"{SAVE_DIR}val_features_360m.csv",   index=False)
test_results_135m.to_csv( f"{SAVE_DIR}test_features_135m.csv",  index=False)
test_results_360m.to_csv( f"{SAVE_DIR}test_features_360m.csv",  index=False)

print(f"All features saved to {SAVE_DIR}")

Mounted at /content/drive
All features saved to /content/drive/MyDrive/mia_features/


# Membership Inference Classifier

In [5]:
# Load pre-extracted features
# Only needed when starting a fresh session without re-running extraction.
# Comment this block out if running the full pipeline top to bottom.

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/mia_features/"

train_results_135m = pd.read_csv(f"{SAVE_DIR}train_features_135m.csv")
train_results_360m = pd.read_csv(f"{SAVE_DIR}train_features_360m.csv")
val_results_135m = pd.read_csv(f"{SAVE_DIR}val_features_135m.csv")
val_results_360m = pd.read_csv(f"{SAVE_DIR}val_features_360m.csv")
test_results_135m = pd.read_csv(f"{SAVE_DIR}test_features_135m.csv")
test_results_360m = pd.read_csv(f"{SAVE_DIR}test_features_360m.csv")

print("Features loaded from disk.")
print(f"  train_135m: {train_results_135m.shape}")
print(f"  train_360m: {train_results_360m.shape}")
print(f"  val_135m:   {val_results_135m.shape}")
print(f"  val_360m:   {val_results_360m.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Features loaded from disk.
  train_135m: (50000, 16)
  train_360m: (50000, 16)
  val_135m:   (10000, 16)
  val_360m:   (10000, 16)


In [22]:
def build_membership_classifier(train_135m, train_360m, val_135m, val_360m):
    """
    Build classifier using full training set and evaluate on validation set.

    """
    print("Building membership inference classifier...")

    # Feature columns
    feature_cols = ['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob',
                    'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio',
                    'min_k10_prob', 'min_k20_prob', 'min_k30_prob']

    # Prepare training features
    train_features_135m = train_135m[feature_cols].copy()
    train_features_360m = train_360m[feature_cols].copy()
    train_features_135m.columns = [f"{col}_135m" for col in train_features_135m.columns]
    train_features_360m.columns = [f"{col}_360m" for col in train_features_360m.columns]
    X_train = pd.concat([train_features_135m, train_features_360m], axis=1)
    y_train = train_135m['label']

    # Prepare validation features
    val_features_135m = val_135m[feature_cols].copy()
    val_features_360m = val_360m[feature_cols].copy()
    val_features_135m.columns = [f"{col}_135m" for col in val_features_135m.columns]
    val_features_360m.columns = [f"{col}_360m" for col in val_features_360m.columns]
    X_val = pd.concat([val_features_135m, val_features_360m], axis=1)
    y_val = val_135m['label']

    # Cross-model difference and ratio features
    # capture how much more confident the 360m model is vs the 135m model
    # since previous runs suggest 360m model has stronger features
    diff_cols = ['loss', 'perplexity', 'min_k10_prob',
                'min_k20_prob', 'min_k30_prob', 'min_token_prob']

    for col in diff_cols:
        X_train[f'{col}_diff']  = X_train[f'{col}_360m'] - X_train[f'{col}_135m']
        X_train[f'{col}_ratio'] = X_train[f'{col}_360m'] / (X_train[f'{col}_135m'] + 1e-10)
        X_val[f'{col}_diff']    = X_val[f'{col}_360m']   - X_val[f'{col}_135m']
        X_val[f'{col}_ratio']   = X_val[f'{col}_360m']   / (X_val[f'{col}_135m'] + 1e-10)

    print(f"Features after adding cross-model diffs: {X_train.shape[1]}")

    # Clean up inf / NaN
    train_medians = X_train.replace([np.inf, -np.inf], np.nan).median()
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(train_medians)
    X_val = X_val.replace([np.inf, -np.inf], np.nan).fillna(train_medians)

    X_train = X_train.fillna(0)
    X_val = X_val.fillna(0)

    all_feature_names = X_train.columns.tolist()
    print(f"Total features before selection: {X_train.shape[1]}")

    selected_features = all_feature_names

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    print(f"\nTraining set:   {X_train.shape[0]} samples")
    print(f"Validation set: {X_val.shape[0]} samples")

    # Train classifiers
    # logistic regression, random forest, XGBoost
    classifiers = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
        ),
    }

    results = {}

    for name, clf in classifiers.items():
        print(f"\nTraining {name}...")

        # Logistic Regression requires scaled features
        # Random Forest and XGBoost operate on raw feature values
        if name == 'Logistic Regression':
            clf.fit(X_train_scaled, y_train)
            y_pred_proba = clf.predict_proba(X_val_scaled)[:, 1]
        else:
            clf.fit(X_train, y_train)
            y_pred_proba = clf.predict_proba(X_val)[:, 1]

        auc = roc_auc_score(y_val, y_pred_proba)

        fpr, tpr, thresholds = roc_curve(y_val, y_pred_proba)
        idx = np.argmin(np.abs(fpr - 0.1))
        tpr_at_fpr_01 = tpr[idx]
        actual_fpr    = fpr[idx]

        y_pred   = (y_pred_proba > 0.5).astype(int)
        accuracy = (y_pred == y_val).mean()

        print(f"  AUC:           {auc:.4f}")
        print(f"  TPR@FPR=0.1:   {tpr_at_fpr_01:.4f}  (actual FPR: {actual_fpr:.4f})")
        print(f"  Accuracy:      {accuracy:.4f}")

        results[name] = {
            'classifier': clf,
            'scaler': scaler if name == 'Logistic Regression' else None,
            'auc': auc,
            'tpr_at_fpr_01': tpr_at_fpr_01,
            'accuracy': accuracy,
            'feature_names': selected_features
        }

        # W&B log classifier metrics
        wandb.log({
            f"{name}/auc":           auc,
            f"{name}/tpr_at_fpr_01": tpr_at_fpr_01,
            f"{name}/accuracy":      accuracy,
        })

        # Feature importances — available for both tree-based models
        if name in ('Random Forest', 'XGBoost'):
            importance_df = pd.DataFrame({
                'feature':    selected_features,
                'importance': clf.feature_importances_
            }).sort_values('importance', ascending=False)

            print(f"\n  Top 10 Most Important Features ({name}, post-selection):")
            for _, row in importance_df.head(10).iterrows():
                print(f"    {row['feature']}: {row['importance']:.4f}")

            # W&B: log feature importances as a bar chart
            wandb.log({
                f"{name}/feature_importances": wandb.plot.bar(
                    wandb.Table(dataframe=importance_df),
                    label="feature",
                    value="importance",
                    title=f"{name} Feature Importances",
                )
            })

    return results

In [23]:
classifier_results = build_membership_classifier(
    train_results_135m, train_results_360m,
    val_results_135m,   val_results_360m
)

Building membership inference classifier...
Features after adding cross-model diffs: 36
Total features before selection: 36

Training set:   50000 samples
Validation set: 10000 samples

Training Logistic Regression...
  AUC:           0.8902
  TPR@FPR=0.1:   0.6626  (actual FPR: 0.1000)
  Accuracy:      0.8208

Training Random Forest...
  AUC:           0.8851
  TPR@FPR=0.1:   0.6668  (actual FPR: 0.1008)
  Accuracy:      0.8083

  Top 10 Most Important Features (Random Forest, post-selection):
    loss_ratio: 0.1329
    perplexity_ratio: 0.0596
    min_k10_prob_diff: 0.0584
    min_k30_prob_diff: 0.0548
    loss_diff: 0.0538
    min_k20_prob_diff: 0.0449
    min_token_prob_360m: 0.0342
    min_k10_prob_360m: 0.0330
    min_k20_prob_ratio: 0.0323
    min_k30_prob_ratio: 0.0294

Training XGBoost...
  AUC:           0.8955
  TPR@FPR=0.1:   0.6688  (actual FPR: 0.1002)
  Accuracy:      0.8116

  Top 10 Most Important Features (XGBoost, post-selection):
    min_k30_prob_diff: 0.1938
    lo

# Results Summary

In [24]:
# Results summary
print("MEMBERSHIP INFERENCE ATTACK RESULTS")
print("=" * 50)

for name, results in classifier_results.items():
    print(f"\n{name}:")
    print(f"  AUC: {results['auc']:.4f}")
    print(f"  TPR@FPR=0.1: {results['tpr_at_fpr_01']:.4f}")
    print(f"  Accuracy: {results['accuracy']:.4f}")

# Best model
best_model = max(classifier_results.values(), key=lambda x: x['auc'])
print(f"\nBEST MODEL PERFORMANCE:")
print(f"AUC: {best_model['auc']:.4f}")
print(f"TPR@FPR=0.1: {best_model['tpr_at_fpr_01']:.4f}")
print(f"Accuracy: {best_model['accuracy']:.4f}")

print(f"\nDataset processed:")
print(f"Training samples: {len(train_results_135m)}")
print(f"Validation samples: {len(val_results_135m)}")
print(f"Total features: {len(best_model['feature_names'])}")

# W&B log best-model summary metrics
best_classifier_name = max(classifier_results.keys(),
                           key=lambda x: classifier_results[x]['auc'])
wandb.summary["best_classifier"] = best_classifier_name
wandb.summary["best_auc"] = best_model['auc']
wandb.summary["best_tpr_at_fpr_01"] = best_model['tpr_at_fpr_01']
wandb.summary["best_accuracy"] = best_model['accuracy']
wandb.summary["n_train_samples"] = len(train_results_135m)
wandb.summary["n_val_samples"] = len(val_results_135m)
wandb.summary["n_features_selected"]= len(best_model['feature_names'])
wandb.summary["selected_features"]  = best_model['feature_names']

MEMBERSHIP INFERENCE ATTACK RESULTS

Logistic Regression:
  AUC: 0.8902
  TPR@FPR=0.1: 0.6626
  Accuracy: 0.8208

Random Forest:
  AUC: 0.8851
  TPR@FPR=0.1: 0.6668
  Accuracy: 0.8083

XGBoost:
  AUC: 0.8955
  TPR@FPR=0.1: 0.6688
  Accuracy: 0.8116

BEST MODEL PERFORMANCE:
AUC: 0.8955
TPR@FPR=0.1: 0.6688
Accuracy: 0.8116

Dataset processed:
Training samples: 50000
Validation samples: 10000
Total features: 36


In [30]:
# Save results and models
import pickle

# Save the best classifier
best_classifier_name = max(classifier_results.keys(), key=lambda x: classifier_results[x]['auc'])
best_classifier = classifier_results[best_classifier_name]

# Save to files
with open('membership_classifier.pkl', 'wb') as f:
    pickle.dump(best_classifier, f)

# Save feature results
train_results_135m.to_csv('train_features_135m.csv', index=False)
train_results_360m.to_csv('train_features_360m.csv', index=False)
val_results_135m.to_csv('val_features_135m.csv', index=False)
val_results_360m.to_csv('val_features_360m.csv', index=False)

print("Results saved:")
print("- membership_classifier.pkl")
print("- train_features_135m.csv")
print("- train_features_360m.csv")
print("- val_features_135m.csv")
print("- val_features_360m.csv")

# W&B: save artefacts so they are versioned alongside the run
artifact = wandb.Artifact(name="membership-classifier", type="model")
artifact.add_file("membership_classifier.pkl")
wandb.log_artifact(artifact)
print("Classifier uploaded to W&B as artifact.")

# HuggingFace Hub: upload classifier and feature CSVs
HF_REPO_ID = "christina-jm/mia-medical-llm"

api = HfApi()
api.create_repo("christina-jm/mia-medical-llm", repo_type="model", exist_ok=True)

# Upload trained classifier
api.upload_file(
    path_or_fileobj="membership_classifier.pkl",
    path_in_repo="membership_classifier.pkl",
    repo_id=HF_REPO_ID,
    repo_type="model",
)

# Upload feature CSVs for reproducibility
for fname in ["train_features_135m.csv", "train_features_360m.csv",
              "val_features_135m.csv",   "val_features_360m.csv"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=f"features/{fname}",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print(f"Classifier and features uploaded to HuggingFace: {HF_REPO_ID}")

Results saved:
- membership_classifier.pkl
- train_features_135m.csv
- train_features_360m.csv
- val_features_135m.csv
- val_features_360m.csv
Classifier uploaded to W&B as artifact.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  membership_classifier.pkl   : 100%|##########|  345kB /  345kB            

Classifier and features uploaded to HuggingFace: christina-jm/mia-medical-llm


# Generate Test Predictions

In [31]:
def prepare_test_predictions(test_135m, test_360m, best_classifier):
    """
    Prepare test features and generate predictions for Kaggle submission.
    """
    print("Preparing test features for prediction...")

    # Feature columns
    feature_cols = ['perplexity', 'loss', 'mean_token_prob', 'min_token_prob', 'max_token_prob',
                   'std_token_prob', 'mean_entropy', 'top_k_mass', 'low_conf_ratio',
                   'min_k10_prob', 'min_k20_prob', 'min_k30_prob']

    # Extract features
    test_features_135m = test_135m[feature_cols].copy()
    test_features_360m = test_360m[feature_cols].copy()

    # Add model suffix
    test_features_135m.columns = [f"{col}_135m" for col in test_features_135m.columns]
    test_features_360m.columns = [f"{col}_360m" for col in test_features_360m.columns]

    # Combine features
    X_test = pd.concat([test_features_135m, test_features_360m], axis=1)

    # feature diff columns
    diff_cols = ['loss', 'perplexity', 'min_k10_prob',
             'min_k20_prob', 'min_k30_prob', 'min_token_prob']

    for col in diff_cols:
        X_test[f'{col}_diff']  = X_test[f'{col}_360m'] - X_test[f'{col}_135m']
        X_test[f'{col}_ratio'] = X_test[f'{col}_360m'] / (X_test[f'{col}_135m'] + 1e-10)

    # Handle infinite/NaN values (use same strategy as training)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(X_test.median())

    # Keep only the features selected during training
    # This ensures test features match the classifier's expected input exactly
    X_test = X_test[best_classifier['feature_names']]

    # Apply scaling if the classifier uses it
    if best_classifier['scaler'] is not None:
        X_test = best_classifier['scaler'].transform(X_test)

    # Generate predictions
    print("Generating predictions on test set...")
    test_predictions = best_classifier['classifier'].predict_proba(X_test)[:, 1]

    return test_predictions

# Get best classifier
best_classifier_name = max(classifier_results.keys(), key=lambda x: classifier_results[x]['auc'])
best_classifier = classifier_results[best_classifier_name]

print(f"Using {best_classifier_name} for test predictions (AUC: {best_classifier['auc']:.4f})")

# Generate test predictions
test_predictions = prepare_test_predictions(test_results_135m, test_results_360m, best_classifier)

print(f"Generated {len(test_predictions)} predictions")
print(f"Prediction range: [{test_predictions.min():.4f}, {test_predictions.max():.4f}]")
print(f"Mean prediction: {test_predictions.mean():.4f}")

Using XGBoost for test predictions (AUC: 0.8955)
Preparing test features for prediction...
Generating predictions on test set...
Generated 15000 predictions
Prediction range: [0.0019, 0.9768]
Mean prediction: 0.4992


In [34]:
# Create Kaggle submission file with correct IDs
print("Creating Kaggle submission file...")

# Use the  'id' column from test dataset
test_ids = test_df['id'].tolist()

# Create submission DataFrame with proper ID mapping
submission_df = pd.DataFrame({
    'id': test_ids,
    'is_member': test_predictions
})

# Save submission file
submission_filename = 'membership_inference_submission.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"Kaggle submission saved: {submission_filename}")
print(f"Shape: {submission_df.shape}")
print(f"ID range: {submission_df['id'].min()} to {submission_df['id'].max()}")
print(f"Prediction range: [{submission_df['is_member'].min():.4f}, {submission_df['is_member'].max():.4f}]")

print("\nFirst 5 rows:")
print(submission_df.head())


# W&B: close the run
wandb.finish()
print("W&B run closed.")

Creating Kaggle submission file...
Kaggle submission saved: membership_inference_submission.csv
Shape: (15000, 2)
ID range: 130000 to 37499
Prediction range: [0.0019, 0.9768]

First 5 rows:
       id  is_member
0   32656   0.939710
1  137254   0.037806
2  134884   0.785888
3   33065   0.679284
4   34304   0.881464


W&B run closed.


In [35]:
from google.colab import files
files.download('membership_inference_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>